# 优化和调参

## torch.optim.lr_scheduler 学习率调度器，用于调整学习率
- 它提供了多种策略来在训练过程中改变优化器的学习率，帮助模型更好地收敛。

In [ ]:
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau

# 标准使用流程
model = nn.Linear(10, 2)
optimizer = SGD(model.parameters(), lr=0.1)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)  # 每10个epoch，lr *= 0.1

for epoch in range(100):
    # 训练...
    train(...)
    
    # 验证...
    validate(...)
    
    scheduler.step()  # 更新学习率（关键！）

## 基于epoch的衰减（无需验证指标）

| 调度器                 | 策略                                 | 适用场景        |
| ------------------- | ---------------------------------- | ----------- |
| `StepLR`            | 每隔 step\_size 个 epoch，lr \*= gamma | 常规训练        |
| `MultiStepLR`       | 在指定 milestones 处衰减                 | 分阶段训练       |
| `ExponentialLR`     | 每个 epoch，lr \*= gamma              | 平滑衰减        |
| `CosineAnnealingLR` | 余弦退火周期调整                           | 现代CNN/ViT训练 |


In [ ]:
# StepLR: 每30个epoch，学习率乘以0.1
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

# MultiStepLR: 在第30、60、80个epoch衰减
scheduler = MultiStepLR(optimizer, milestones=[30, 60, 80], gamma=0.1)

# CosineAnnealingLR: T_max为半周期长度，eta_min为最小lr
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

## 基于验证指标的衰减（需要监控性能）
| 调度器                           | 策略        | 特点      |
| ----------------------------- | --------- | ------- |
| `ReduceLROnPlateau`           | 验证指标停滞时衰减 | 最灵活，最常用 |
| `CosineAnnealingWarmRestarts` | 带热重启的余弦退火 | 逃离局部最优  |


In [ ]:
# 当验证loss 5个epoch不下降，lr *= 0.5
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min',      # 'min' 监控loss, 'max' 监控acc
    factor=0.5,      # 衰减系数
    patience=5,      # 容忍epoch数
    verbose=True     # 打印学习率变化
)

# 训练循环中需要传入验证指标
val_loss = validate(...)
scheduler.step(val_loss)  # 注意：传入监控值！

## 高级策略： 预热 + 余弦退火（transformer标配）

In [ ]:
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR
import math

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """带线性预热的余弦退火调度器（Hugging Face风格）"""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    return LambdaLR(optimizer, lr_lambda)

# 使用
scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=500, 
    num_training_steps=10000
)

## OneCycleLR（超收敛策略）

In [ ]:
# 先上升到max_lr，再下降到base_lr以下
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.1,
    total_steps=1000,  # 或 epochs * steps_per_epoch
    pct_start=0.3,     # 上升阶段占比
    anneal_strategy='cos',  # 'cos' 或 'linear'
    div_factor=25.,    # 初始lr = max_lr/25
    final_div_factor=1e4   # 最终lr = 初始lr/1e4
)

In [ ]:
model = MyModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# 组合策略：预热 + 余弦退火
scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=1000,
    num_training_steps=total_steps
)

for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        # 前向/反向/优化
        loss = ...
        loss.backward()
        optimizer.step()
        scheduler.step()  # 每个step更新（OneCycleLR需要）
        optimizer.zero_grad()
    
    # 验证
    val_metric = evaluate(model, val_loader)
    
    # 可选：基于指标的调度
    # plateau_scheduler.step(val_metric)

## 注意事项

| 要点        | 说明                                                  |
| --------- | --------------------------------------------------- |
| **调用位置**  | `scheduler.step()` 放在 epoch 结束或 batch 结束（取决于策略）     |
| **顺序**    | `optimizer.step()` → `scheduler.step()`（新版本PyTorch） |
| **状态保存**  | 保存 `scheduler.state_dict()` 以恢复训练                   |
| **学习率读取** | `optimizer.param_groups[0]['lr']`                   |


In [ ]:
# 保存完整状态
checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'scheduler': scheduler.state_dict(),  # 别忘了！
    'epoch': epoch
}
torch.save(checkpoint, 'checkpoint.pth')

# torch.optim.RMSprop: 它是 AdaGrad 的改进版，通过引入移动平均来解决学习率单调递减的问题，非常适合处理非平稳目标和递归神经网络（RNN）
- RMSprop 的核心思想是：为每个参数自适应地调整学习率，基于该参数历史梯度的平方的移动平均。

# 特殊用途函数和实用工具

## torch.distributed 支持分布式训练的模块
- torch.distributed 是 PyTorch 提供的分布式计算后端，支持多种通信后端（NCCL、Gloo、MPI），用于多 GPU、多节点的并行训练。


In [ ]:
import torch.distributed as dist

# 初始化进程组
dist.init_process_group(
    backend='nccl',      # 通信后端: nccl(推荐GPU)/gloo/cpu
    init_method='env://', # 初始化方式: env://(环境变量)/tcp://(指定地址)/file://
    world_size=4,        # 总进程数
    rank=0               # 当前进程ID (0 ~ world_size-1)
)

# 销毁进程组
dist.destroy_process_group()


### 常用后端对比

| 后端       | 适用场景       | 特点          |
| -------- | ---------- | ----------- |
| **NCCL** | NVIDIA GPU | 最优性能，支持多机多卡 |
| **Gloo** | CPU / GPU  | 通用性强，支持各种设备 |
| **MPI**  | 高性能集群      | 需要预装MPI库    |


### 核心通信原语

In [ ]:
import torch
import torch.distributed as dist

# 假设已初始化，world_size=4
tensor = torch.tensor([1.0]).cuda()

# 1. 广播 (Broadcast) - 从rank0广播到所有rank
dist.broadcast(tensor, src=0)

# 2. 规约 (Reduce) - 求和后存到rank0
dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)

# 3. 全规约 (All-Reduce) - 求和后分发到所有rank ⭐最常用
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)

# 4. 全收集 (All-Gather) - 收集所有rank的数据
tensor_list = [torch.zeros(1).cuda() for _ in range(4)]
dist.all_gather(tensor_list, tensor)

# 5. 散射 (Scatter) - 从rank0分发数据
dist.scatter(tensor, scatter_list=None, src=0)

# 6. 点对点发送/接收
if dist.get_rank() == 0:
    dist.send(tensor, dst=1)
else:
    dist.recv(tensor, src=0)

### 实际应用：DDP (DistributedDataParallel)

In [ ]:
import torch
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP

# 初始化
torch.cuda.set_device(local_rank)
dist.init_process_group(backend='nccl')

# 包装模型
model = MyModel().to(local_rank)
ddp_model = DDP(
    model,
    device_ids=[local_rank],
    output_device=local_rank,
    find_unused_parameters=False  # 动态图需设为True
)

# 训练（与普通模型用法相同）
output = ddp_model(input)
loss = criterion(output, target)
loss.backward()  # 梯度自动同步
optimizer.step()

### 启动： 单节点+ 多GPU

In [ ]:
# 使用 torchrun (推荐)
torchrun --nproc_per_node=4 train.py

# 或使用 python -m torch.distributed.launch (旧版)
python -m torch.distributed.launch --nproc_per_node=4 train.py

### 启动： 多节点+ 多GPU

In [ ]:
# Node 0 (主节点)
torchrun \
    --nnodes=2 \
    --node_rank=0 \
    --nproc_per_node=8 \
    --master_addr="192.168.1.1" \
    --master_port=12345 \
    train.py

# Node 1
torchrun \
    --nnodes=2 \
    --node_rank=1 \
    --nproc_per_node=8 \
    --master_addr="192.168.1.1" \
    --master_port=12345 \
    train.py

### 关键工具函数

In [ ]:
# 获取当前环境信息
dist.get_rank()       # 当前进程ID
dist.get_world_size() # 总进程数
dist.is_initialized() # 是否已初始化

# 同步屏障（所有进程到达后才继续）
dist.barrier()

# 检查后端支持
dist.is_nccl_available()
dist.is_gloo_available()
dist.is_mpi_available()

| 特性                               | 说明                                  |
| -------------------------------- | ----------------------------------- |
| **torch.distributed.fsdp**       | Fully Sharded Data Parallel，大模型分片训练 |
| **torch.distributed.rpc**        | 远程过程调用，支持参数服务器架构                    |
| **torch.distributed.elastic**    | 弹性训练，支持动态扩缩容                        |
| **torch.distributed.checkpoint** | 分布式模型检查点保存/加载                       |


### 使用流程
1. 设置环境变量 (RANK, WORLD_SIZE, MASTER_ADDR等)
   ↓
2. dist.init_process_group() 初始化
   ↓
3. 使用 DDP 包装模型 或 手动调用通信原语
   ↓
4. 训练循环
   ↓
5. dist.destroy_process_group() 清理

## torch.no_grad() 是一个禁用梯度计算的上下文管理器，主要用于推理阶段或不需要反向传播的操作。

### 核心机制

| 特性              | 说明                             |
| --------------- | ------------------------------ |
| **禁用 Autograd** | 不创建计算图，不跟踪操作历史                 |
| **节省内存**        | 不保存中间激活值，显存占用大幅降低              |
| **加速计算**        | 跳过梯度相关计算，速度更快                  |
| **不影响参数**       | 参数仍可修改（如通过 `optimizer.step()`） |


### 上下文管理器（推荐）

In [ ]:
import torch

model = torch.nn.Linear(10, 5)
input = torch.randn(2, 10)

with torch.no_grad():
    output = model(input)  # 不计算梯度
    # 内部所有操作都不追踪梯度

### 与 requires_grad 的区别

In [ ]:
import torch

# 场景1: 创建不追踪梯度的张量
x = torch.tensor([1.0], requires_grad=False)  # 永久不追踪

# 场景2: 临时禁用梯度（推荐用于推理）
x = torch.tensor([1.0], requires_grad=True)
with torch.no_grad():
    y = x * 2  # 不追踪
z = x * 3      # 追踪（退出上下文后恢复）

| 方式                    | 作用域      | 用途        |
| --------------------- | -------- | --------- |
| `requires_grad=False` | 张量级别，永久  | 冻结参数、输入数据 |
| `torch.no_grad()`     | 代码块级别，临时 | 推理、验证、测试  |


### 实际应用场景
- 场景1: 模型推理/评估

In [ ]:
model.eval()  # 设置评估模式（影响 BatchNorm/Dropout）
with torch.no_grad():  # 禁用梯度
    for data, target in test_loader:
        output = model(data)
        loss = criterion(output, target)
        # 不调用 loss.backward()，不更新参数

- 场景2: 计算指标（不训练）

In [ ]:
with torch.no_grad():
    accuracy = (pred.argmax(dim=1) == target).float().mean()
    # 纯计算，不需要梯度

- 场景3: 冻结部分网络

In [ ]:
# 方式1: 直接设置 requires_grad
for param in model.backbone.parameters():
    param.requires_grad = False

# 方式2: 使用 no_grad 上下文（仅推理时）
with torch.no_grad():
    features = model.backbone(input)  # 不计算 backbone 的梯度
output = model.head(features)         # head 部分计算梯度
loss.backward()  # 只更新 head 的参数

### torch.no_grad() vs torch.set_grad_enabled()

In [ ]:
# 方式1: 上下文管理器（推荐）
with torch.no_grad():
    ...

# 方式2: 全局开关（影响整个程序）
torch.set_grad_enabled(False)  # 全局禁用
# ... 推理代码 ...
torch.set_grad_enabled(True)   # 恢复

# 方式3: 条件控制
is_train = False
with torch.set_grad_enabled(is_train):
    output = model(input)

### 与 detach() 的关系

In [ ]:
x = torch.tensor([1.0], requires_grad=True)

# 方式1: no_grad 上下文
with torch.no_grad():
    y = x * 2  # y 无 grad_fn，但 x 的 requires_grad 不变

# 方式2: detach()
y = (x * 2).detach()  # 返回新张量，与原计算图分离

# 关键区别
with torch.no_grad():
    y = x * 2
# y.requires_grad = False, 但 x 仍追踪梯度

z = x.detach()
# z 是 x 的视图，共享内存，但 z.requires_grad = False

| 特性   | `no_grad()` | `detach()`  |
| ---- | ----------- | ----------- |
| 作用范围 | 代码块内所有操作    | 单个张量        |
| 内存   | 不共享         | 共享数据内存      |
| 使用场景 | 推理、评估       | 需要张量值但阻断梯度流 |


### 性能对比示例

In [3]:
import torch
import time

model = torch.nn.Linear(1000, 1000).cuda()
x = torch.randn(64, 1000).cuda()

# 有梯度
torch.cuda.synchronize()
start = time.time()
for _ in range(1000):
    out = model(x)
    loss = out.sum()
    loss.backward()
torch.cuda.synchronize()
print(f"With grad: {time.time() - start:.3f}s")  # 慢，内存占用高

# 无梯度
model.zero_grad()
torch.cuda.synchronize()
start = time.time()
with torch.no_grad():
    for _ in range(1000):
        out = model(x)
torch.cuda.synchronize()
print(f"No grad: {time.time() - start:.3f}s")    # 快，内存占用低

AssertionError: Torch not compiled with CUDA enabled

✅ 评估/测试时必须使用: model.eval() + torch.no_grad()
✅ 计算指标、可视化、日志记录时使用
✅ 推理部署时使用
✅ 需要临时阻断梯度传播时使用

❌ 训练前向传播时不要使用（无法反向传播）
❌ 需要梯度检查点 (gradient checkpointing) 时不要使用

# 数学和统计
- torch.mean()
- torch.std()
- torch.var()
- torch.exp()
- torch.log()
- torch.sin()
- torch.cos()

# 张量变换和操作